# Tooth Caries Classification (PyTorch Lightning + W&B)

This notebook performs binary classification on cropped tooth images using:

- Existing project data-preparation modules
- A custom PyTorch Lightning training loop
- Weights & Biases logging for loss, accuracy, and F1-metrics

In [ ]:
from __future__ import annotations

import os
import sys
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Any
import json

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader

from torchvision.models import resnet50, ResNet50_Weights

import lightning as L
from lightning.pytorch.callbacks import ModelCheckpoint, LearningRateMonitor, EarlyStopping
from lightning.pytorch.loggers import WandbLogger
import torchmetrics

import wandb
from dotenv import load_dotenv
from wandb.errors import AuthenticationError

PROJECT_ROOT = Path('/work')
SCRIPTS_ROOT = PROJECT_ROOT / 'scripts'
CKPT_PATH = "last-v8.ckpt"

for p in (PROJECT_ROOT, SCRIPTS_ROOT):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

from classification_pipeline import (
    load_or_download_classification_dataset,
    ClassificationDownloadConfig,
    build_multiclass_classification_records_from_masks,
    split_grouped_records,
    ToothCropDataset,
    #build_classification_image_pipeline,
    build_classification_resize_pipeline
)

In [4]:
@dataclass
class TrainConfig:
    image_size: int = 224
    batch_size: int = 32
    num_workers: int = 4
    max_epochs: int = 200
    lr: float = 3e-4
    weight_decay: float = 1e-4
    wandb_project: str = 'tooth-caries-classification'
    wandb_run_name: str = 'efficientnet-full-dataset-v1-full run-7 label'
    force_download: bool = False

cfg = TrainConfig()
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)

Device: cuda


In [ ]:
class ToothClassificationDataModule(L.LightningDataModule):
    def __init__(self, cfg: TrainConfig):
        super().__init__()
        self.cfg = cfg
        self.train_ds = None
        self.val_ds = None
        self.test_ds = None

    def setup(self, stage: str | None = None):
        _, coco_data, image_dirs = load_or_download_classification_dataset(
            ClassificationDownloadConfig(api_key=os.getenv('ROBOFLOW_API_KEY')),
            force_download=self.cfg.force_download
        )
        
        all_records, self.label_map = build_multiclass_classification_records_from_masks(
            coco_data, image_dirs, crop_margin=0.08
        )

        MACRO_CLASS_MAPPING = {
            44: 0,
            36: 1, 45: 1, 53: 1,
            72: 2, 74: 2, 75: 2,
            47: 3,
            1: 4, 34: 4, 35: 4, 37: 4, 38: 4, 39: 4, 40: 4, 41: 4, 42: 4, 43: 4, 46: 4,
            48: 4, 49: 4, 50: 4, 51: 4, 52: 4, 54: 4, 55: 4, 56: 4, 57: 4, 58: 4, 59: 4,
            60: 4, 63: 4, 64: 4, 65: 4, 66: 4, 67: 4, 68: 4, 69: 4, 70: 4, 71: 4, 76: 4, 77: 4, 78: 4, 79: 4,
            73: 5,
            61: 6, 62: 6, 0: 6
        }
        for i in range(2, 34): MACRO_CLASS_MAPPING[i] = 6

        print("Osztályok összevonása 7 fő kategóriára...")
        for rec in all_records:
            old_label = rec['label']
            rec['label'] = MACRO_CLASS_MAPPING.get(old_label, 6)

        self.label_map = {
            'Caries': 0, 'Filling': 1, 'Endodontics': 2,
            'Crown': 3, 'Implant': 4, 'Retained_Root': 5, 'Healthy': 6
        }
        
        train_rec, val_rec, test_rec = split_grouped_records(
            all_records, train_size=0.7, val_size=0.15, test_size=0.15
        )

        all_labels = [rec['label'] for rec in all_records]
        num_classes = len(self.label_map)
        total_samples = len(all_labels)

        weights = []
        for i in range(num_classes):
            count = all_labels.count(i)

            weight = total_samples / (num_classes * count) if count > 0 else 0.0
            weights.append(weight)

        self.class_weights = torch.tensor(weights, dtype=torch.float)

        print(f"Number of classes: {num_classes}")
        print(f'Class weights: {self.class_weights}')

        self.train_ds = train_rec
        self.val_ds = val_rec
        self.test_ds = test_rec
        
        aug_pipeline = build_classification_image_pipeline()
        resize_pipeline = build_classification_resize_pipeline(self.cfg.image_size)
        
        self.train_ds = ToothCropDataset(
            self.train_ds,
            image_size=self.cfg.image_size,
            image_transform=aug_pipeline,
            resize_transform=resize_pipeline,
            output_channels=3
        )
        self.val_ds = ToothCropDataset(
            self.val_ds,
            image_size=self.cfg.image_size,
            resize_transform=resize_pipeline,
            output_channels=3
        )
        self.test_ds = ToothCropDataset(
            self.test_ds,
            image_size=self.cfg.image_size,
            resize_transform=resize_pipeline,
            output_channels=3
        )
        
        print(f'Train samples: {len(self.train_ds)}\nVal samples: {len(self.val_ds)}\nTest samples: {len(self.test_ds)}')
    
    def _loader_kwargs(self) -> dict[str, Any]:
        num_workers = int(self.cfg.num_workers)
        kwargs: dict[str, Any] = {
            'num_workers': num_workers,
            'pin_memory': torch.cuda.is_available(),
        }
        if num_workers > 0:
            kwargs['persistent_workers'] = True
            kwargs['prefetch_factor'] = 2
        return kwargs

    def train_dataloader(self):
        return DataLoader(
            self.train_ds,
            batch_size=self.cfg.batch_size,
            shuffle=True,
            **self._loader_kwargs()
        )

    def val_dataloader(self):
        return DataLoader(
            self.val_ds,
            batch_size=self.cfg.batch_size,
            shuffle=False,
            **self._loader_kwargs()
        )
        
    def test_dataloader(self):
        return DataLoader(
            self.test_ds,
            batch_size=self.cfg.batch_size,
            shuffle=False,
            **self._loader_kwargs()
        )

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import lightning as L
import torchmetrics
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights

class LitToothClassifier(L.LightningModule):
    def __init__(self, cfg, num_classes: int = 7, class_weights=None):
        super().__init__()

        self.cfg = cfg

        self.model = efficientnet_b0(weights=EfficientNet_B0_Weights.DEFAULT)

        for param in self.model.parameters():
            print(param.requires_grad)

        if class_weights is not None:
            self.register_buffer('class_weights', class_weights)
        else:
            self.class_weights = None

        in_features = self.model.classifier[1].in_features
        self.model.classifier[1] = nn.Linear(in_features, num_classes)

        self.accuracy = torchmetrics.Accuracy(task="multiclass", num_classes=num_classes)
        self.f1 = torchmetrics.F1Score(task="multiclass", num_classes=num_classes, average="macro")

    def forward(self, x):
        return self.model(x)

    def training_step(self, batch, batch_idx):
        x, y, _ = batch
        logits = self(x)
        loss = F.cross_entropy(logits, y, weight=self.class_weights)

        self.log("train/loss", loss, prog_bar=True, on_step=False, on_epoch=True)
        return loss

    def validation_step(self, batch, batch_idx):
        x, y, _ = batch
        logits = self(x)
        loss = F.cross_entropy(logits, y, weight=self.class_weights)

        preds = torch.argmax(logits, dim=1)
        self.accuracy(preds, y)
        self.f1(preds, y)

        self.log("val/loss", loss, prog_bar=True, on_epoch=True)
        self.log("val/acc", self.accuracy, prog_bar=True, on_epoch=True)
        self.log("val/f1", self.f1, prog_bar=True, on_epoch=True)
        return loss

    def test_step(self, batch, batch_idx):
        x, y, _ = batch

        logits = self(x)

        loss = F.cross_entropy(logits, y, weight=self.class_weights if hasattr(self, 'class_weights') else None)

        preds = torch.argmax(logits, dim=1)
        self.accuracy(preds, y)
        self.f1(preds, y)

        self.log("test/loss", loss, on_step=False, on_epoch=True)
        self.log("test/acc", self.accuracy, on_step=False, on_epoch=True)
        self.log("test/f1", self.f1, on_step=False, on_epoch=True)

        return loss

    def configure_optimizers(self):
      optimizer = torch.optim.AdamW(
          filter(lambda p: p.requires_grad, self.parameters()),
          lr=self.cfg.lr,
          weight_decay=self.cfg.weight_decay
      )
      return optimizer

In [10]:
env_path = PROJECT_ROOT / '.env'
if not env_path.exists():
    raise FileNotFoundError(f'.env file not found at {env_path}')

load_dotenv(env_path, override=True)

wandb_secret = (os.getenv('WANDB_API_KEY') or '').strip().strip('"').strip("'")
if not wandb_secret:
    raise RuntimeError(f'WANDB_API_KEY not found in {env_path}')

if len(wandb_secret) < 20:
    raise RuntimeError(
        f'Invalid WANDB_API_KEY length ({len(wandb_secret)}). '
        'Expected a classic API key or a wandb_v1 access token from https://wandb.ai/authorize.'
    )

# Support both classic API keys and wandb_v1 access tokens.
try:
    os.environ['WANDB_API_KEY'] = wandb_secret
    wandb.login(key=wandb_secret, relogin=True)
except AuthenticationError:
    if wandb_secret.startswith('wandb_v1_'):
        # Compatibility fallback for SDKs that validate only classic key shapes.
        compat_key = wandb_secret.replace('wandb_v1_', '', 1)
        os.environ['WANDB_API_KEY'] = compat_key
        wandb.login(key=compat_key, relogin=True)
    else:
        raise

print('W&B login successful using WANDB_API_KEY from .env')

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: nagycsd01 (nagycsd01-university-of-budapest-technology-and-economics) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


W&B login successful using WANDB_API_KEY from .env


In [11]:
datamodule = ToothClassificationDataModule(cfg)
datamodule.setup()

Betöltés memóriába innen: /work/ready_crops/pre_cropped_records.json
🔄 Osztályok összevonása 7 fő kategóriára...
Number of classes: 7
Class weights: tensor([  2.6389,   3.7181,  76.3749,  10.8879,   0.2326, 130.4738,   0.5152])
Train samples: 76625
Val samples: 16593
Test samples: 16380


In [12]:
datamodule.label_map

{'Caries': 0,
 'Filling': 1,
 'Endodontics': 2,
 'Crown': 3,
 'Implant': 4,
 'Retained_Root': 5,
 'Healthy': 6}

In [ ]:
model = LitToothClassifier(cfg=cfg, num_classes=len(datamodule.label_map), class_weights=datamodule.class_weights)

wandb_logger = WandbLogger(
    project=cfg.wandb_project,
    name=cfg.wandb_run_name,
    log_model=True,
    id='1knvdeyx',
    resume='must'
)

checkpoint_cb = ModelCheckpoint(
    dirpath=str(PROJECT_ROOT / 'output' / 'checkpoints' / 'classification'),
    filename='full run-efficientnet-7-classes',
    monitor='val/f1',
    mode='max',
    save_top_k=2,
    save_last=True
)

early_stop_cb = EarlyStopping(
    monitor='val/f1',
    patience=10,
    mode='max'
)

trainer = L.Trainer(
    max_epochs=cfg.max_epochs,
    accelerator='gpu',
    devices=1,
    precision='16-mixed' if torch.cuda.is_available() else 32,
    accumulate_grad_batches=4,
    logger=wandb_logger,
    callbacks=[
        checkpoint_cb,
        LearningRateMonitor(logging_interval='epoch'),
        early_stop_cb
    ],
    log_every_n_steps=5,
    check_val_every_n_epoch=1
)

trainer.fit(model, datamodule=datamodule, ckpt_path=CKPT_PATH)

wandb.finish()

Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth
100%|██████████████████████████████████████████████████████████████████████████████| 20.5M/20.5M [00:01<00:00, 11.8MB/s]
Using 16bit Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
You are using a CUDA device ('NVIDIA GeForce RTX 3050 Ti Laptop GPU') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float

True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True


wandb: WARNING The anonymous setting has no effect and will be removed in a future version.


Betöltés memóriába innen: /work/ready_crops/pre_cropped_records.json
🔄 Osztályok összevonása 7 fő kategóriára...
Number of classes: 7
Class weights: tensor([  2.6389,   3.7181,  76.3749,  10.8879,   0.2326, 130.4738,   0.5152])
Train samples: 76625
Val samples: 16593
Test samples: 16380


Checkpoint directory /work/output/checkpoints/classification exists and is not empty.
Restoring states from the checkpoint path at last-v8.ckpt
You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.


...

Trying to infer the `batch_size` from an ambiguous collection. The batch size we found is 32. To avoid any 
miscalculations, use `self.log(..., batch_size=batch_size)`.

Trying to infer the `batch_size` from an ambiguous collection. The batch size we found is 17. To avoid any 
miscalculations, use `self.log(..., batch_size=batch_size)`.


Detected KeyboardInterrupt, attempting graceful shutdown ...


SystemExit: 1

To exit: use 'exit', 'quit', or Ctrl-D.


In [14]:
wandb.finish()

epoch,▁▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇████
lr-AdamW,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train/loss,██▆▅▄▆▄▃▄▄▃▂▄▁▂▃
trainer/global_step,▁▁▁▁▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▇▇▇▇█████
val/acc,▅▄▄▂▁▆█▃▄▄▄▅▁▆▅▃
val/f1,▃▁▁▆▆▆▅▁▅▃▇▂█▄▆▁
val/loss,▁▂▃▁▂█▃▆▃▃▂▅▃▆▆▃
epoch,69
lr-AdamW,0.0003
train/loss,0.32482
trainer/global_step,41930
